# MiniCode v2 — fine-tuning (QLoRA) do gerador de especificações

**Hipótese.** Após o fine-tuning com QLoRA, o Qwen2.5-Coder-1.5B-Instruct passa a gerar especificações que o
agente aceita já na primeira tentativa (`validar_spec`) e que cumprem critérios de qualidade (critérios de aceitação
verificáveis, exemplos concretos, sem código de implementação, só biblioteca padrão), numa taxa próxima à do Qwen-14B.

**Comparação (nível 1).** mesmo modelo antes × depois do treino, com o Qwen-14B como referência.
**Nível 2.** o modelo treinado como gerador de spec dentro do MiniCode (seção final).

Pré-requisitos: GPU (Ambiente de execução → T4) e o repositório com `dataset_tratado/{train,val,test}.jsonl`
gerado por `python gerar_datasets.py`. Se o repositório for privado, use um token na URL do `git clone`.

In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.12" accelerate bitsandbytes
import os, sys
REPO = "https://github.com/VLFAri/MiniCode.git"
if not os.path.exists("MiniCode"):
    !git clone -q {REPO}
%cd MiniCode
sys.path.insert(0, os.getcwd())

In [ ]:
import torch
assert torch.cuda.is_available(), "Ative a GPU: Ambiente de execução > Alterar tipo de ambiente > T4"

MODELO      = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DATA_DIR    = "dataset_tratado"
MAX_LEN     = 2048          # exemplos maiores são DESCARTADOS (cortar a spec ensinaria specs incompletas)
MAX_NEW     = 1400          # tokens gerados na avaliação
USAR_QLORA  = True          # base em 4 bits (NF4); False = base em fp32
R, ALPHA, DROPOUT = 16, 32, 0.05
EPOCAS, LR, BATCH, ACUM = 4, 2e-4, 2, 4
SEED = 42
AVALIAR_COM_TENTATIVAS = False   # True mede o GeradorSpec com até 3 tentativas (3x mais lento)

## 1. Dataset (e checagem de que ele está no formato do agente)

In [ ]:
import json, random
import numpy as np
from datasets import load_dataset
from src.spec.qualidade import validar_spec_estrita
from src.spec.gerador import PROMPT_SPEC

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
arquivos = {"train": f"{DATA_DIR}/train.jsonl", "validation": f"{DATA_DIR}/val.jsonl", "test": f"{DATA_DIR}/test.jsonl"}
ds = load_dataset("json", data_files=arquivos)
print({k: len(v) for k, v in ds.items()})

# Se isto falhar, o dataset está no formato errado: regere com `python gerar_datasets.py`
for split in ds:
    for ex in ds[split]:
        assert ex["messages"][0]["content"] == PROMPT_SPEC.format(tarefa=ex["task"], contexto=ex["contexto"]), "prompt diferente do agente"
        assert not validar_spec_estrita(ex["messages"][1]["content"]), "spec fora do padrão"
stats = json.load(open(f"{DATA_DIR}/stats.json"))
print("válidos:", stats["itens_validos"], "/", stats["itens"], "| 1ª tentativa válida do teacher:", stats["teacher_primeira_tentativa_valida"])
print(ds["train"][0]["messages"][1]["content"][:700])

## 2. Tokenização

O prompt (idêntico ao do agente) recebe `label = -100`, ignorado pela loss; só a spec + EOS é treinada.
`tokenizar_exemplo` usa o mesmo `apply_chat_template` da inferência. Exemplos acima de `MAX_LEN` são descartados.

In [ ]:
from transformers import AutoTokenizer
from src.spec.treino import tokenizar_exemplo

tok = AutoTokenizer.from_pretrained(MODELO)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

tokenizado = ds.map(lambda ex: tokenizar_exemplo(tok, ex["messages"], MAX_LEN), remove_columns=ds["train"].column_names)
antes = {k: len(v) for k, v in tokenizado.items()}
tokenizado = tokenizado.filter(lambda ex: not ex["truncado"]).remove_columns("truncado")
print("descartados por tamanho:", {k: antes[k] - len(tokenizado[k]) for k in antes})
tam = [len(x) for x in tokenizado["train"]["input_ids"]]
print(f"tokens por exemplo: média={np.mean(tam):.0f} p90={np.percentile(tam, 90):.0f} máx={max(tam)}")
ex = tokenizado["train"][0]
print("começa a loss em:", repr(tok.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100][:25])))

## 3. Modelo base e configuração QLoRA/LoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if USAR_QLORA:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(MODELO, quantization_config=bnb, device_map={"": 0})
    model = prepare_model_for_kbit_training(model)          # gradient checkpointing + camadas de norma em fp32
else:
    model = AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype=torch.float32).to("cuda")
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()

lora = LoraConfig(r=R, lora_alpha=ALPHA, lora_dropout=DROPOUT, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)        # ATENÇÃO: sem esta linha o treino não treinaria o adapter
model.print_trainable_parameters()

### Por que LoRA treina tão poucos parâmetros

Em vez de atualizar `W ∈ ℝ^{d×d}`, congelamos `W` e aprendemos `ΔW = BA` com `B ∈ ℝ^{d×r}`, `A ∈ ℝ^{r×d}` e
`r ≪ d`; a saída é `W' = W + (α/r)·BA`. `W` tem `d²` parâmetros e `BA` tem `2rd`. `B` começa em zero, então **antes
do treino o modelo com LoRA é idêntico ao base**: a avaliação da próxima seção é o baseline sem recarregar nada.
No QLoRA, `W` fica congelado em 4 bits (NF4) e só `A` e `B` são treinados em 16 bits.

In [ ]:
d = model.config.hidden_size
print(f"d={d}, r={R}: q_proj tem {d*d:,} parâmetros; ΔW=BA treina {2*R*d:,} ({100*2*R*d/(d*d):.2f}%)")

## 4. Baseline: o MESMO modelo antes do fine-tuning (nível 1)

In [ ]:
from src.spec.modelo_local import LocalSpecModel
from src.spec.avaliacao import avaliar_modelo, carregar_teste

teste = carregar_teste(f"{DATA_DIR}/test.jsonl")
model.config.use_cache = True
gerador = LocalSpecModel(model=model, tokenizer=tok, max_new_tokens=MAX_NEW)
m_base = avaliar_modelo(gerador, teste, com_tentativas=AVALIAR_COM_TENTATIVAS)
print(json.dumps({k: (round(v, 3) if isinstance(v, float) else v) for k, v in m_base.items() if k != "problemas_mais_comuns"}, indent=1))
print("problemas mais comuns:", m_base["problemas_mais_comuns"])

## 5. Treinamento

Treino por épocas (sem `max_steps`): com ~100 exemplos e batch efetivo 8, são ~12 passos por época. Avaliação de
validação a cada época e, no fim, é carregado o checkpoint de menor `eval_loss` (reduz o risco de overfitting).

In [ ]:
import inspect
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

bf16 = torch.cuda.is_bf16_supported()
chave = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"
args = TrainingArguments(
    output_dir="saida_treino", num_train_epochs=EPOCAS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH, gradient_accumulation_steps=ACUM,
    learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=0.05, logging_steps=5,
    optim="paged_adamw_8bit" if USAR_QLORA else "adamw_torch",
    fp16=not bf16, bf16=bf16, save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    report_to="none", seed=SEED, **{chave: "epoch"},
)
trainer = Trainer(model=model, args=args, train_dataset=tokenizado["train"], eval_dataset=tokenizado["validation"],
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
model.config.use_cache = False
trainer.train()

In [ ]:
import matplotlib.pyplot as plt

os.makedirs("experiments/results", exist_ok=True)
hist = trainer.state.log_history
json.dump(hist, open("experiments/results/spec_log_history.json", "w"), indent=1)
treino = [(h["epoch"], h["loss"]) for h in hist if "loss" in h]
valid = [(h["epoch"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(7, 4))
plt.plot(*zip(*treino), alpha=.7, label="loss de treino (a cada 5 passos)")
plt.plot(*zip(*valid), "o-", label="loss de validação (por época)")
plt.xlabel("época"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=.3)
plt.savefig("experiments/results/spec_loss.png", dpi=150, bbox_inches="tight"); plt.show()

## 6. Avaliação: base × fine-tuned × Qwen-14B (nível 1)

In [ ]:
model.config.use_cache = True
gerador = LocalSpecModel(model=model, tokenizer=tok, max_new_tokens=MAX_NEW)
m_ft = avaliar_modelo(gerador, teste, com_tentativas=AVALIAR_COM_TENTATIVAS)

# Referência: Qwen-14B com o prompt REAL do agente (sem o system privilegiado). Chave em Colab > Secrets.
# Se o endpoint estiver fora do ar, esta célula só avisa e segue. Depois, localmente (MINICODE_BACKEND=local):
#   python experiments/eval_spec.py --modo teacher   -> o relatório usa experiments/results/spec_teacher.json
try:
    from google.colab import userdata
    os.environ["MINICODE_API_KEY"] = userdata.get("MINICODE_API_KEY")
    from src.modelo.cliente_llm import LLMClient
    m_14b = avaliar_modelo(LLMClient(max_tokens=2048), teste, com_tentativas=AVALIAR_COM_TENTATIVAS)
except Exception as e:
    print("referência Qwen-14B não avaliada:", e)
    m_14b = None

linhas = [("modelo base (baseline)", m_base), ("base + QLoRA (fine-tuned)", m_ft), ("Qwen-14B (referência)", m_14b)]
print(f"{'modelo':<28}{'válida p/ agente':>17}{'válida estrita':>16}{'critérios':>11}{'c/ código':>11}")
for nome, m in linhas:
    if m:
        print(f"{nome:<28}{m['valida_agente']:>17.2f}{m['valida_estrita']:>16.2f}{m['criterios_medio']:>11.1f}{m['com_codigo_de_implementacao']:>11.2f}")
print(f"n_teste = {m_base['n']}  (com poucos exemplos, diferenças pequenas não são conclusivas)")
json.dump({"base": m_base, "finetuned": m_ft, "qwen14b": m_14b,
           "config": dict(MODELO=MODELO, R=R, ALPHA=ALPHA, EPOCAS=EPOCAS, LR=LR, QLORA=USAR_QLORA)},
          open("experiments/results/spec_colab.json", "w"), indent=2, ensure_ascii=False)

In [ ]:
# Exemplo qualitativo: a mesma tarefa, depois do treino
ex = teste[0]
print("TAREFA:", ex["task"], "\nCONTEXTO:", ex["contexto"], "\n")
saida = gerador.complete(ex["messages"][0]["content"])
print(saida[:1800])
print("\nproblemas (validação estrita):", validar_spec_estrita(saida))

## 7. Salvar o adapter

In [ ]:
model.save_pretrained("adapter_spec")
tok.save_pretrained("adapter_spec")
!zip -qr adapter_spec.zip adapter_spec
from google.colab import files
files.download("adapter_spec.zip")

## 8. Nível 2: o modelo treinado dentro do agente

Descompacte `adapter_spec/` no repositório (no Colab, com GPU) e compare **as mesmas tarefas inéditas** com o
Qwen-14B gerando a spec e com o modelo pequeno (a flag `--spec-4bit` reproduz o QLoRA do treino):

```
python src/cli/main.py --avaliar tarefas_teste --saida experiments/res_spec_14b
python src/cli/main.py --avaliar tarefas_teste --spec-model base --spec-4bit --saida experiments/res_spec_base
python src/cli/main.py --avaliar tarefas_teste --spec-model lora --spec-adapter adapter_spec --spec-4bit --saida experiments/res_spec_lora
```

Compare `HiddenSuccessRate`, `SpecFirstTryValid`, `AverageSpecAttempts` e `AverageLLMCalls` nos três
`resultados.json`. Só a especificação muda de modelo; plano, código e reparo continuam no Qwen-14B. Melhorar a
qualidade da spec não garante melhorar o agente: reporte o que medir, inclusive se não houver ganho.